In [1]:

# ============================================================
# DATA DISCRETIZATION - ALL MAJOR TECHNIQUES
# Banking Customer Dataset
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 1. CREATE BANKING DATASET
# ------------------------------------------------------------

data = {
    'Customer_ID': range(101, 121),

    'Age': [22, 35, 28, 45, 32, 40, 25, 50, 38, 29,
            42, 31, 55, 27, 48, 36, 33, 44, 26, 39],

    'Salary': [25000, 45000, 30000, 65000, 40000,
               55000, 28000, 80000, 50000, 32000,
               60000, 38000, 90000, 27000, 75000,
               48000, 42000, 68000, 29000, 52000],

    'Balance': [10000, 25000, 15000, 50000, 20000,
                35000, 12000, 70000, 30000, 18000,
                45000, 22000, 85000, 11000, 65000,
                28000, 24000, 55000, 13000, 40000],

    'Loan_Amount': [0, 50000, 20000, 100000, 30000,
                    60000, 0, 150000, 40000, 10000,
                    80000, 25000, 200000, 0, 120000,
                    45000, 35000, 90000, 0, 70000],

    'Transactions': [10, 25, 15, 40, 20, 30, 12, 50, 28, 18,
                     35, 22, 60, 11, 45, 27, 24, 42, 14, 32],

    'Gender': ['F', 'M', 'F', 'M', 'M', 'F', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'F', 'M', 'F', 'M', 'M', 'F', 'M']
}

df = pd.DataFrame(data)

print("ORIGINAL BANKING DATASET")
display(df)


# ------------------------------------------------------------
# 2. EQUAL WIDTH BINNING
# ------------------------------------------------------------

# Divides the complete data range into equal-width intervals.
# Example: Age is divided into 3 intervals.

df_equal_width = df.copy()

df_equal_width['Age_Category'] = pd.cut(
    df_equal_width['Age'],
    bins=3,
    labels=['Low', 'Medium', 'High']
)

print("\n2. EQUAL WIDTH BINNING")
display(df_equal_width[['Age', 'Age_Category']])


# ------------------------------------------------------------
# 3. EQUAL FREQUENCY BINNING
# ------------------------------------------------------------

# Divides records into groups containing approximately
# the same number of observations.

df_equal_frequency = df.copy()

df_equal_frequency['Age_Category'] = pd.qcut(
    df_equal_frequency['Age'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

print("\n3. EQUAL FREQUENCY BINNING")
display(df_equal_frequency[['Age', 'Age_Category']])


# ------------------------------------------------------------
# 4. CUSTOM BINNING
# ------------------------------------------------------------

# Creates intervals according to user-defined boundaries.

df_custom = df.copy()

df_custom['Age_Group'] = pd.cut(
    df_custom['Age'],
    bins=[0, 18, 30, 45, 60],
    labels=['Teenager', 'Young', 'Adult', 'Senior']
)

print("\n4. CUSTOM BINNING")
display(df_custom[['Age', 'Age_Group']])


# ------------------------------------------------------------
# 5. BINNING WITH LABELS
# ------------------------------------------------------------

# Converts numerical salary values into meaningful categories.

df_labels = df.copy()

df_labels['Salary_Category'] = pd.cut(
    df_labels['Salary'],
    bins=[-1, 30000, 60000, 100000, float('inf')],
    labels=['Low Salary', 'Medium Salary',
            'High Salary', 'Very High Salary']
)

print("\n5. BINNING WITH LABELS")
display(df_labels[['Salary', 'Salary_Category']])


# ------------------------------------------------------------
# 6. UNSUPERVISED DISCRETIZATION
# ------------------------------------------------------------

# KBinsDiscretizer automatically divides numerical data
# into a specified number of bins.
# Strategy can be uniform, quantile or kmeans.

age_data = df[['Age']]

# Uniform strategy = Equal Width Binning

uniform_model = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',
    strategy='uniform'
)

df_uniform = df.copy()

df_uniform['Age_Bin'] = uniform_model.fit_transform(
    age_data
).astype(int)

print("\n6A. UNSUPERVISED - UNIFORM")
display(df_uniform[['Age', 'Age_Bin']])


# Quantile strategy = Equal Frequency Binning

quantile_model = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',
    strategy='quantile'
)

df_quantile = df.copy()

df_quantile['Age_Bin'] = quantile_model.fit_transform(
    age_data
).astype(int)

print("\n6B. UNSUPERVISED - QUANTILE")
display(df_quantile[['Age', 'Age_Bin']])


# KMeans strategy = Cluster-based Binning

kmeans_model = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',
    strategy='kmeans'
)

df_kmeans_bins = df.copy()

df_kmeans_bins['Age_Bin'] = kmeans_model.fit_transform(
    age_data
).astype(int)

print("\n6C. UNSUPERVISED - KMEANS STRATEGY")
display(df_kmeans_bins[['Age', 'Age_Bin']])


# ------------------------------------------------------------
# 7. K-MEANS BASED DISCRETIZATION
# ------------------------------------------------------------

# Groups similar age values into clusters.
# Each cluster represents one age category.

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df_cluster = df.copy()

df_cluster['Age_Cluster'] = kmeans.fit_predict(
    df[['Age']]
)

print("\n7. K-MEANS BASED DISCRETIZATION")
display(df_cluster[['Age', 'Age_Cluster']])

print("Cluster Centers:")
print(kmeans.cluster_centers_)


# ------------------------------------------------------------
# 8. DECISION TREE BASED DISCRETIZATION
# ------------------------------------------------------------

# Uses a target variable to find useful age intervals.
# Example: Predict whether a customer takes a loan.
# This is supervised discretization.

df_tree = df.copy()

df_tree['Loan_Status'] = (
    df_tree['Loan_Amount'] > 0
).astype(int)

X = df_tree[['Age']]
y = df_tree['Loan_Status']

tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

tree.fit(X, y)

# Predict the class for each customer.

df_tree['Age_Category'] = tree.predict(X)

print("\n8. DECISION TREE BASED DISCRETIZATION")
display(df_tree[['Age', 'Loan_Status', 'Age_Category']])

# Displays the age boundaries learned by the tree.

print("Decision Tree Split Points:")
print(sorted([
    value
    for value in tree.tree_.threshold
    if value != -2
]))


# ------------------------------------------------------------
# 9. ENTROPY-BASED DISCRETIZATION
# ------------------------------------------------------------

# Uses information gain to identify useful split points.
# Decision trees use entropy to find boundaries.

entropy_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=2,
    random_state=42
)

entropy_tree.fit(X, y)

df_entropy = df.copy()

df_entropy['Age_Category'] = entropy_tree.predict(X)

print("\n9. ENTROPY-BASED DISCRETIZATION")
display(df_entropy[['Age', 'Loan_Amount', 'Age_Category']])

print("Entropy-Based Split Points:")
print(sorted([
    value
    for value in entropy_tree.tree_.threshold
    if value != -2
]))


# ------------------------------------------------------------
# 10. CHI-SQUARE BASED DISCRETIZATION
# ------------------------------------------------------------

# Chi-square tests the relationship between numerical
# features (after binning) and a categorical target.
# It is used here to evaluate discretized features.

df_chi = df.copy()

df_chi['Loan_Status'] = (
    df_chi['Loan_Amount'] > 0
).astype(int)

# First convert Age into categories.

df_chi['Age_Bin'] = pd.qcut(
    df_chi['Age'],
    q=3,
    labels=False
)

# Convert the categorical bins into one-hot columns.

age_dummies = pd.get_dummies(
    df_chi['Age_Bin'],
    prefix='Age_Bin',
    dtype=int
)

# Chi-square requires non-negative feature values.

chi_scores, p_values = chi2(
    age_dummies,
    df_chi['Loan_Status']
)

chi_results = pd.DataFrame({
    'Age_Bin': age_dummies.columns,
    'Chi_Square_Score': chi_scores,
    'P_Value': p_values
})

print("\n10. CHI-SQUARE EVALUATION")
display(chi_results)


# ------------------------------------------------------------
# 11. CONCEPT HIERARCHY GENERATION
# ------------------------------------------------------------

# Converts detailed values into higher-level categories.
# Example:
# Hyderabad, Chennai, Mumbai, Delhi -> Indian Cities
# Young, Adult, Senior -> Age Groups

df_hierarchy = df.copy()

# City hierarchy

city_hierarchy = {
    'Hyderabad': 'South India',
    'Chennai': 'South India',
    'Mumbai': 'West India',
    'Delhi': 'North India'
}

df_hierarchy['Region'] = df_hierarchy['Gender'].map(
    lambda x: 'Not Applicable'
)

df_hierarchy['Region'] = df_hierarchy['City'].map(
    city_hierarchy
)

# Age hierarchy

df_hierarchy['Age_Group'] = pd.cut(
    df_hierarchy['Age'],
    bins=[0, 30, 45, 100],
    labels=['Young', 'Middle Age', 'Senior']
)

print("\n11. CONCEPT HIERARCHY GENERATION")
display(
    df_hierarchy[
        ['Age', 'Age_Group', 'City', 'Region']
    ]
)


# ------------------------------------------------------------
# 12. DISCRETIZATION OF MULTIPLE FEATURES
# ------------------------------------------------------------

# Converts Age, Salary and Balance into categories.

df_multiple = df.copy()

# Age categories

df_multiple['Age_Group'] = pd.cut(
    df_multiple['Age'],
    bins=[0, 30, 45, 100],
    labels=['Young', 'Adult', 'Senior']
)

# Salary categories

df_multiple['Salary_Group'] = pd.cut(
    df_multiple['Salary'],
    bins=[-1, 30000, 60000, float('inf')],
    labels=['Low', 'Medium', 'High']
)

# Balance categories

df_multiple['Balance_Group'] = pd.cut(
    df_multiple['Balance'],
    bins=[-1, 20000, 50000, float('inf')],
    labels=['Low', 'Medium', 'High']
)

print("\n12. MULTIPLE FEATURE DISCRETIZATION")

display(
    df_multiple[
        ['Age', 'Age_Group',
         'Salary', 'Salary_Group',
         'Balance', 'Balance_Group']
    ]
)


# ------------------------------------------------------------
# 13. FINAL COMPARISON
# ------------------------------------------------------------

print("\n========================================")
print("   FINAL DATA DISCRETIZATION SUMMARY")
print("========================================")

print("Original Dataset Shape:", df.shape)

print("Equal Width Shape:", df_equal_width.shape)

print("Equal Frequency Shape:", df_equal_frequency.shape)

print("Custom Binning Shape:", df_custom.shape)

print("Uniform Discretization Shape:", df_uniform.shape)

print("Quantile Discretization Shape:", df_quantile.shape)

print("KMeans Discretization Shape:", df_kmeans_bins.shape)

print("Decision Tree Shape:", df_tree.shape)

print("Entropy-Based Shape:", df_entropy.shape)

print("Concept Hierarchy Shape:", df_hierarchy.shape)

print("\nALL DATA DISCRETIZATION TECHNIQUES COMPLETED!")


ORIGINAL BANKING DATASET


,Customer_ID,Age,Salary,Balance,Loan_Amount,Transactions,Gender
0,101,22,25000,10000,0,10,F
1,102,35,45000,25000,50000,25,M
2,103,28,30000,15000,20000,15,F
3,104,45,65000,50000,100000,40,M
4,105,32,40000,20000,30000,20,M
5,106,40,55000,35000,60000,30,F
6,107,25,28000,12000,0,12,F
7,108,50,80000,70000,150000,50,M
8,109,38,50000,30000,40000,28,F
9,110,29,32000,18000,10000,18,M



2. EQUAL WIDTH BINNING


,Age,Age_Category
0,22,Low
1,35,Medium
2,28,Low
3,45,High
4,32,Low
5,40,Medium
6,25,Low
7,50,High
8,38,Medium
9,29,Low



3. EQUAL FREQUENCY BINNING


,Age,Age_Category
0,22,Low
1,35,Medium
2,28,Low
3,45,High
4,32,Medium
5,40,High
6,25,Low
7,50,High
8,38,Medium
9,29,Low


... (15 outputs left out)

,Age,Loan_Amount,Age_Category
0,22,0,0
1,35,50000,1
2,28,20000,1
3,45,100000,1
4,32,30000,1
5,40,60000,1
6,25,0,0
7,50,150000,1
8,38,40000,1
9,29,10000,1


Entropy-Based Split Points:
[np.float64(27.5)]

10. CHI-SQUARE EVALUATION


,Age_Bin,Chi_Square_Score,P_Value
0,Age_Bin_0,6.035714,0.014019
1,Age_Bin_1,1.500000,0.220671
2,Age_Bin_2,1.750000,0.185877


KeyError: 'City'